# Multi-Agent News Brief

Find current news, have an editor filter it, have a writer create a script, then generate an AI voice briefing.

## 1. Install dependencies and configure your OpenAI key

In [ ]:
%pip install -q openai-agents ddgs

import os

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")

## 2. Find and inspect current news items

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException


def search_news(query: str) -> str:
    """Find recent news and return the title, summary, date, and URL of each item."""
    try:
        results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    except DDGSException as error:
        raise RuntimeError(f"News search failed: {error}") from error

    if not results:
        raise RuntimeError("No recent news results were found. Try a different topic.")

    print(f"News query: {query}")
    print(f"Items found: {len(results)}")
    for index, item in enumerate(results, start=1):
        print(f"\n{index}. {item.get('title', 'Untitled')}")
        print(item.get('date', 'No date available.'))
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\n"
        f"Date: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\n"
        f"URL: {item.get('url', 'No URL available.')}"
        for item in results
    )


topic = "artificial intelligence business"
news_items = search_news(topic)

## 3. Editor Agent: filter and refine the news

In [ ]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="""
You are a careful news editor. Select the three most relevant, credible, and non-duplicative items.
Exclude advertisements, speculation, clickbait, and items without useful evidence.
For every selected item, preserve its title, key facts, publication date, and URL.
If the sources are insufficient, say so.
""",
)

editor_result = await Runner.run(
    starting_agent=editor_agent,
    input=f"Topic: {topic}\n\nNews items:\n{news_items}",
)
edited_news = editor_result.final_output
print(edited_news)

## 4. Writer Agent: create a short news script

In [ ]:
writer_agent = Agent(
    name="News Script Writer",
    instructions="""
Write a neutral 45- to 60-second spoken news script based only on the editor's selected items.
Use clear, natural language. Do not add unsupported facts.
End by naming the source publications, but do not read full URLs aloud.
""",
)

writer_result = await Runner.run(
    starting_agent=writer_agent,
    input=f"Create a spoken news script from this edited brief:\n\n{edited_news}",
)
news_script = writer_result.final_output
print(news_script)

## 5. Generate the audio briefing

This creates AI-generated speech. Disclose that the voice is AI-generated when sharing the audio.

In [ ]:
from IPython.display import Audio, display
from openai import AsyncOpenAI

tts_client = AsyncOpenAI()
speech = await tts_client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=news_script,
    instructions="Speak clearly in a calm, neutral broadcast-news style.",
)

audio_path = "news_brief.mp3"
await speech.write_to_file(audio_path)
display(Audio(audio_path))

## 6. Download the MP3

Run this cell to save the generated audio file to your computer.

In [ ]:
from google.colab import files

files.download(audio_path)